# Cross-Game Comparison Analysis

In [ ]:
import pathlibimport osimport matplotlib.pyplot as pltimport pandas as pdimport seaborn as snssns.set_theme(style="whitegrid")if not pathlib.Path("pyproject.toml").exists():    os.chdir(pathlib.Path("../.."))RESULTS_DIR = pathlib.Path("results")MODEL = "gpt-5-mini"

## Part 1: Standard Personas Comparison

In [ ]:
PERSONAS = ["normal", "strategic", "greedy", "benevolent"]FIGURES_DIR = pathlib.Path("analysis/figures/comparison")FIGURES_DIR.mkdir(parents=True, exist_ok=True)GAMES = {    "ultimatum": ("results/ultimatum", "ultimatum_1.csv"),    "dictator": ("results/dictator", "dictator_1.csv"),    "dictator_anonymous": ("results/dictator_anonymous", "dictator_anonymous_1.csv"),}dfs = []for game, (results_dir, csv_name) in GAMES.items():    for persona in PERSONAS:        path = pathlib.Path(results_dir) / persona / MODEL / csv_name        if path.exists():            tmp = pd.read_csv(path)            tmp["game"] = game            dfs.append(tmp)        else:            print(f"Not found: {path}")df = pd.concat(dfs, ignore_index=True)print(f"Total rows: {len(df)}")df.groupby(["game", "persona"])["offer"].describe()

### Violin: Persona x Game

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))sns.violinplot(data=df, x="persona", y="offer", hue="game", order=PERSONAS,    hue_order=list(GAMES), inner="box", ax=ax)ax.set_title("Offer Distribution by Persona and Game")ax.set_xlabel("Persona")ax.set_ylabel("Offer")ax.legend(title="Game")fig.tight_layout()fig.savefig(FIGURES_DIR / "violin_persona_by_game.png", dpi=150)plt.show()

### Violin: Gender x Game (by Persona)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10), sharey=True)for ax, persona in zip(axes.flat, PERSONAS):    subset = df[df["persona"] == persona]    sns.violinplot(data=subset, x="gender", y="offer", hue="game",        hue_order=list(GAMES), inner="box", ax=ax)    ax.set_title(persona)    ax.set_xlabel("Gender")    ax.set_ylabel("Offer")    ax.legend(title="Game", fontsize=8, title_fontsize=9)fig.suptitle("Offer Distribution by Gender and Game", fontsize=14)fig.tight_layout()fig.savefig(FIGURES_DIR / "violin_gender_by_game.png", dpi=150)plt.show()

## Part 2: Big Five Personas Comparison

In [ ]:
PERSONAS_BIG5 = ["openness", "conscientiousness", "extraversion", "agreeableness", "neuroticism"]FIGURES_DIR_BIG5 = pathlib.Path("analysis/figures/comparison_big5")FIGURES_DIR_BIG5.mkdir(parents=True, exist_ok=True)dfs_b5 = []for game, (results_dir, csv_name) in GAMES.items():    for persona in PERSONAS_BIG5:        path = pathlib.Path(results_dir) / persona / MODEL / csv_name        if path.exists():            tmp = pd.read_csv(path)            tmp["game"] = game            dfs_b5.append(tmp)        else:            print(f"Not found: {path}")df_b5 = pd.concat(dfs_b5, ignore_index=True)print(f"Total rows: {len(df_b5)}")df_b5.groupby(["game", "persona"])["offer"].describe()

### Violin: Persona x Game (Big Five)

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))sns.violinplot(data=df_b5, x="persona", y="offer", hue="game", order=PERSONAS_BIG5,    hue_order=list(GAMES), inner="box", ax=ax)ax.set_title("Offer Distribution by Persona and Game (Big Five)")ax.set_xlabel("Persona")ax.set_ylabel("Offer")ax.legend(title="Game")fig.tight_layout()fig.savefig(FIGURES_DIR_BIG5 / "violin_persona_by_game.png", dpi=150)plt.show()

### Violin: Gender x Game (by Persona, Big Five)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10), sharey=True)for ax, persona in zip(axes.flat, PERSONAS_BIG5):    subset = df_b5[df_b5["persona"] == persona]    sns.violinplot(data=subset, x="gender", y="offer", hue="game",        hue_order=list(GAMES), inner="box", ax=ax)    ax.set_title(persona)    ax.set_xlabel("Gender")    ax.set_ylabel("Offer")    ax.legend(title="Game", fontsize=8, title_fontsize=9)axes.flat[-1].set_visible(False)fig.suptitle("Offer Distribution by Gender and Game (Big Five)", fontsize=14)fig.tight_layout()fig.savefig(FIGURES_DIR_BIG5 / "violin_gender_by_game.png", dpi=150)plt.show()

## Part 3: Cross-Game Gender Analysis

In [ ]:
FIGURES_DIR_GENDER = pathlib.Path(f"analysis/figures/cross_game/{MODEL}")FIGURES_DIR_GENDER.mkdir(parents=True, exist_ok=True)# Load receiver datareceiver_dfs = []for source in ["human", "computer"]:    subdir = "unfair_receiver" if source == "human" else "unfair_receiver_computer"    for persona in PERSONAS:        path = RESULTS_DIR / subdir / persona / MODEL / "unfair_receiver_1.csv"        if path.exists():            tmp = pd.read_csv(path)            tmp["game"] = f"unfair_receiver_{source}"            receiver_dfs.append(tmp)        else:            print(f"Not found: {path}")df_recv = pd.concat(receiver_dfs, ignore_index=True)df_recv["decision"] = df_recv["decision"].str.strip().str.lower()df_recv["accepted"] = (df_recv["decision"] == "accept").astype(int)# Use df (proposer data) from Part 1 as df_propdf_prop = df.copy()df_prop["offer"] = pd.to_numeric(df_prop["offer"], errors="coerce")print(f"Proposer rows: {len(df_prop)}, Receiver rows: {len(df_recv)}")

### Mean Offer by Game x Gender

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))pivot_plot = df_prop.groupby(["game", "gender"])["offer"].mean().unstack()pivot_plot.plot(kind="bar", ax=ax)ax.set_title("Mean Offer by Game × Gender", fontsize=14)ax.set_xlabel("Game")ax.set_ylabel("Mean Offer ($)")ax.legend(title="Gender")for container in ax.containers:    ax.bar_label(container, fmt="%.1f", fontsize=8)plt.xticks(rotation=15, ha="right")fig.tight_layout()fig.savefig(FIGURES_DIR_GENDER / "mean_offer_by_game_gender.png", dpi=150)plt.show()

### Accept Rate by Proposal Source x Gender

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))pivot_plot2 = df_recv.groupby(["game", "gender"])["accepted"].mean().unstack()pivot_plot2.plot(kind="bar", ax=ax)ax.set_title("Accept Rate by Proposal Source × Gender", fontsize=14)ax.set_xlabel("Proposal Source")ax.set_ylabel("Accept Rate")ax.set_ylim(0, 1)ax.legend(title="Gender")for container in ax.containers:    ax.bar_label(container, fmt="%.2f", fontsize=8)plt.xticks(rotation=0)fig.tight_layout()fig.savefig(FIGURES_DIR_GENDER / "accept_rate_by_source_gender.png", dpi=150)plt.show()

### Faceted by Persona: Mean Offer

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10), sharey=True)for ax, persona in zip(axes.flat, PERSONAS):    sub = df_prop[df_prop["persona"] == persona]    pv = sub.groupby(["game", "gender"])["offer"].mean().unstack()    pv.plot(kind="bar", ax=ax)    ax.set_title(f"Persona: {persona}", fontsize=12)    ax.set_xlabel("")    ax.set_ylabel("Mean Offer ($)")    ax.legend(title="Gender", fontsize=8)    for container in ax.containers:        ax.bar_label(container, fmt="%.1f", fontsize=7)    ax.tick_params(axis="x", rotation=15)fig.suptitle("Mean Offer by Game × Gender (by Persona)", fontsize=14)fig.tight_layout()fig.savefig(FIGURES_DIR_GENDER / "mean_offer_by_game_gender_persona.png", dpi=150)plt.show()

### Faceted by Persona: Accept Rate

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10), sharey=True)for ax, persona in zip(axes.flat, PERSONAS):    sub = df_recv[df_recv["persona"] == persona]    pv = sub.groupby(["game", "gender"])["accepted"].mean().unstack()    pv.plot(kind="bar", ax=ax)    ax.set_title(f"Persona: {persona}", fontsize=12)    ax.set_xlabel("")    ax.set_ylabel("Accept Rate")    ax.set_ylim(0, 1)    ax.legend(title="Gender", fontsize=8)    for container in ax.containers:        ax.bar_label(container, fmt="%.2f", fontsize=7)    ax.tick_params(axis="x", rotation=0)fig.suptitle("Accept Rate by Source × Gender (by Persona)", fontsize=14)fig.tight_layout()fig.savefig(FIGURES_DIR_GENDER / "accept_rate_by_source_gender_persona.png", dpi=150)plt.show()

### Grand Summary

In [ ]:
offer_summary = df_prop.groupby(["gender", "game"])["offer"].mean().unstack().round(2)offer_summary.columns = [f"{c} (offer)" for c in offer_summary.columns]accept_summary = df_recv.groupby(["gender", "game"])["accepted"].mean().unstack()accept_summary = accept_summary.map(lambda x: f"{x:.1%}")accept_summary.columns = [f"{c} (accept%)" for c in accept_summary.columns]grand = pd.concat([offer_summary, accept_summary], axis=1)grand